# 01 · Local Left latest-partition check

Run this notebook once per day from a host that can reach the Left databases and the Databricks SQL warehouse.

The Delta table is the only source of truth. The latest row for each pair supplies the current configuration. This notebook skips pairs whose human-maintained `review_status` is `CHECKED_SUCCEEDED`, queries the latest Left partition and its row count, and appends one immutable daily check row. It never changes `review_status`.

Requires `databricks-sql-connector >= 3.0` for native `?` parameter markers. Schedule it to finish before the Databricks Right-check job starts.


In [ ]:
import os

TABLE_LIST = os.environ.get("DTRACK_TABLE_LIST", "<catalog>.<schema>.table_list")
WAREHOUSE_HTTP_PATH = os.environ.get("DATABRICKS_HTTP_PATH", "/sql/1.0/warehouses/<warehouse-id>")
SERVER_HOSTNAME = os.environ.get("DATABRICKS_SERVER_HOSTNAME", "<workspace>.cloud.databricks.com")
ACCESS_TOKEN = os.environ.get("DATABRICKS_TOKEN")
RUN_DATE = None                 # None = today in TABLE timezone; "YYYY-MM-DD" to backfill
PAIR_NAMES = None               # None = every active pair; for example ["accounts"]
DRY_RUN = False


In [ ]:
import json
import re
import uuid
from datetime import UTC, date, datetime
from zoneinfo import ZoneInfo

from databricks import sql as dbsql

_TABLE = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*$")
if not _TABLE.fullmatch(TABLE_LIST):
    raise ValueError("TABLE_LIST must be catalog.schema.table")
if not ACCESS_TOKEN:
    raise ValueError("Set DATABRICKS_TOKEN; do not store credentials in the table or notebook")

warehouse = dbsql.connect(
    server_hostname=SERVER_HOSTNAME.removeprefix("https://"),
    http_path=WAREHOUSE_HTTP_PATH,
    access_token=ACCESS_TOKEN,
)


In [ ]:
LATEST_CONFIG_SQL = f"""
WITH ranked AS (
  SELECT *,
         row_number() OVER (
           PARTITION BY pair_name
           ORDER BY config_version DESC, created_at DESC, check_id DESC
         ) AS rn
  FROM {TABLE_LIST}
)
SELECT pair_name, config_version, timezone, review_status, offset_days, aws_name,
       left_query_variable, right_query_variable,
       left_conn_detail, right_conn_detail
FROM ranked
WHERE rn = 1
  AND review_status <> 'CHECKED_SUCCEEDED'
ORDER BY pair_name
"""

with warehouse.cursor() as cursor:
    cursor.execute(LATEST_CONFIG_SQL)
    names = [column[0] for column in cursor.description]
    configs = [dict(zip(names, row)) for row in cursor.fetchall()]

if PAIR_NAMES:
    selected = set(PAIR_NAMES)
    configs = [row for row in configs if row["pair_name"] in selected]

print(f"{len(configs)} active pair(s):", [row["pair_name"] for row in configs])


In [ ]:
def parse_detail(value):
    detail = json.loads(value) if isinstance(value, str) else dict(value)
    forbidden = {"password", "token", "secret", "dsn"}

    def check_keys(item):
        if isinstance(item, dict):
            if any(str(key).lower() in forbidden for key in item):
                raise ValueError("Connection JSON must contain logical references only, never credentials or DSNs")
            for nested in item.values():
                check_keys(nested)
        elif isinstance(item, list):
            for nested in item:
                check_keys(nested)

    check_keys(detail)
    return detail


def connect_left(detail):
    source = detail.get("source", "oracle").lower()
    if source == "oracle":
        import oracledb

        name = detail["conn_name"]
        return oracledb.connect(
            user=os.environ[f"{name}_USR"],
            password=os.environ[f"{name}_PWD"],
            dsn=os.environ[f"{name}_DSN"],
        )
    if source == "duckdb":
        import duckdb

        return duckdb.connect(detail["path"], read_only=True)
    raise ValueError(f"Unsupported Left source: {source}")


def relation(detail):
    return f"({detail['query']}) source_query" if detail.get("query") else detail["table"]


def partition_expr(source, variable, partition_type):
    if partition_type == "date":
        return f"TRUNC({variable})" if source == "oracle" else f"CAST({variable} AS DATE)"
    return variable


def sql_literal(value, partition_type):
    text = str(value)[:10] if partition_type == "date" else str(value)
    if partition_type == "date":
        return f"DATE '{text}'"
    if partition_type in {"int", "numeric"}:
        return text
    return "'" + text.replace("'", "''") + "'"


def normalize_partition(value, partition_type):
    if value is None:
        return None
    if partition_type == "date":
        return value.date().isoformat() if isinstance(value, datetime) else str(value)[:10]
    return str(value)


def left_latest(config):
    detail = parse_detail(config["left_conn_detail"])
    source = detail.get("source", "oracle").lower()
    partition_type = detail.get("partition_type", "date")
    expression = partition_expr(source, config["left_query_variable"], partition_type)
    filters = [detail["where"]] if detail.get("where") else []
    where = (" WHERE " + " AND ".join(f"({item})" for item in filters)) if filters else ""
    connection = connect_left(detail)
    try:
        cursor = connection.cursor()
        cursor.execute(f"SELECT MAX({expression}) FROM {relation(detail)}{where}")
        latest = cursor.fetchone()[0]
        if latest is None:
            raise ValueError("latest Left partition is empty")
        literal = sql_literal(latest, partition_type)
        cursor.execute(
            f"SELECT COUNT(*) FROM {relation(detail)}"
            + (" WHERE " if not filters else where + " AND ")
            + f"({expression} = {literal})"
        )
        count = int(cursor.fetchone()[0])
        return normalize_partition(latest, partition_type), count
    finally:
        connection.close()


In [ ]:
MERGE_SQL = f"""
MERGE INTO {TABLE_LIST} AS target
USING (VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)) AS source (
  check_id, pair_name, config_version, query_date, timezone, review_status, offset_days, aws_name,
  left_query_variable, right_query_variable, left_conn_detail, right_conn_detail,
  left_partition, left_count, count_state, left_checked_at, error_message, created_at, updated_at
)
ON target.check_id = source.check_id
WHEN MATCHED AND target.count_state = 'ERROR' THEN UPDATE SET
  left_partition = source.left_partition, left_count = source.left_count,
  count_state = source.count_state, left_checked_at = source.left_checked_at,
  error_message = source.error_message, updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT (
  check_id, pair_name, config_version, query_date, timezone, review_status, offset_days, aws_name,
  left_query_variable, right_query_variable, left_conn_detail, right_conn_detail,
  left_partition, right_partition, left_count, right_count, count_state,
  left_checked_at, right_checked_at, error_message, created_at, updated_at
) VALUES (
  source.check_id, source.pair_name, source.config_version, source.query_date, source.timezone,
  source.review_status, source.offset_days, source.aws_name, source.left_query_variable,
  source.right_query_variable, source.left_conn_detail, source.right_conn_detail,
  source.left_partition, NULL, source.left_count, NULL, source.count_state,
  source.left_checked_at, NULL, source.error_message, source.created_at, source.updated_at
)
"""

records = []
for config in configs:
    zone = ZoneInfo(config["timezone"] or "America/New_York")
    query_date = date.fromisoformat(RUN_DATE) if RUN_DATE else datetime.now(zone).date()
    checked_at = datetime.now(UTC)
    error = None
    try:
        left_partition, left_count = left_latest(config)
        state = "PENDING_RIGHT"
    except Exception as exc:
        left_partition, left_count = None, None
        state = "ERROR"
        error = f"{type(exc).__name__}: {exc}"

    check_id = str(uuid.uuid5(
        uuid.NAMESPACE_URL,
        f"{TABLE_LIST}:{config['pair_name']}:{config['config_version']}:{query_date.isoformat()}",
    ))
    record = (
        check_id, config["pair_name"], int(config["config_version"]), query_date, config["timezone"],
        config["review_status"], int(config["offset_days"] or 0), config["aws_name"],
        config["left_query_variable"], config["right_query_variable"],
        config["left_conn_detail"], config["right_conn_detail"],
        left_partition, left_count, state, checked_at, error, checked_at, checked_at,
    )
    records.append(record)

if DRY_RUN:
    for record in records:
        print(record[:6], record[11:16])
else:
    with warehouse.cursor() as cursor:
        for record in records:
            cursor.execute(MERGE_SQL, record)

print(f"{'Prepared' if DRY_RUN else 'Merged'} {len(records)} daily check row(s)")
